# Music Annotation Project — Notebook 40
## V8.3 Residual-Gap Continuation Feasibility Audit

**Runtime:** CPU. Run only after Notebook 39 semantic review is complete.

### Scientific question
Among the human-reviewed Notebook 39 regions, what proportion of pixels admitted at `p_print >= 0.50` but not at `p_print >= 0.75` lie on musical structures for which continuation is geometrically plausible, rather than on isolated or semantically unsuitable ink?

### Correction to the hypothesis
The historical implementation did **not** implement generic completion of stems, slurs or hairpins. It implemented:

- Telea, Navier–Stokes and biharmonic crop inpainting;
- a conservative horizontal `safe_staff_continuation` rule;
- two hybrids that combined biharmonic/Navier–Stokes with that staff continuation.

Notebook 40 therefore separates:

1. **old logic actually implemented** — horizontal staff continuation;
2. **plausible but not historically implemented** — stems/barlines, beams, slurs/ties and hairpins;
3. **not a continuation target** — compact glyphs, annotation/non-print and indeterminate cases.

It does not silently claim that the old code can complete every semantically continuable structure.

### V8.3 experiment
For thresholds 0.75 and 0.50, selected observed pixels are removed from the residual reconstruction mask and remain exact observed bytes. The historical five-method reconstruction benchmark is then rerun on the residual mask:

- `telea`
- `navier_stokes`
- `biharmonic`
- `hybrid_conservative`
- `hybrid_routed`

The exact historical staff continuation constants are retained: horizontal opening width 35, evidence dilation 3×7, maximum gap 80, 16-pixel flank search, and the 35th-percentile dark value capped at 140.

### Guardrails
- development pages only;
- Notebook 39 semantic labels must be complete;
- no retraining, recalibration or threshold adoption;
- clean truth is loaded only for scoring reconstructed outputs;
- outside-mask pixels are immutable;
- selected threshold pixels must remain exact observed bytes;
- V7.8 and all source artefacts are read-only and hash checked;
- V8.3 is a new reconstruction experiment and has no production or promotion authority.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas numpy pyarrow matplotlib openpyxl

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,math,platform,sys,shutil,zipfile
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt
from skimage.restoration import inpaint as sk_inpaint
from skimage.morphology import skeletonize
from skimage.metrics import structural_similarity as ssim

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB39_ROOT=ROOT/'observed_print_classifier_v8_2j_reconstruction_sensitivity'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_3_residual_gap_continuation'
SOURCE_NB39_RUN_OVERRIDE=None
HIGH_THRESHOLD=.75;LOW_THRESHOLD=.50;MIN_PRINT_RUN=2
CROP_MARGIN=48;MAX_STAFF_GAP=80;BOOTSTRAP_SAMPLES=5000;SEED=42
METHODS=['telea','navier_stokes','biharmonic','hybrid_conservative','hybrid_routed']

IMPLEMENTED_OLD={'STAFF_LINE'}
PLAUSIBLE_UNIMPLEMENTED={'STEM_OR_BARLINE','BEAM','SLUR_OR_TIE','CRESCENDO_DECRESCENDO_HAIRPIN'}
NOT_CONTINUATION={'NOTEHEAD_OR_COMPACT_GLYPH','OTHER_PRINTED_SYMBOL'}
NON_TARGET={'ANNOTATION_OR_NON_PRINT','INDETERMINATE','MIXED_PRINTED_STRUCTURES'}

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_3_residual_gap_continuation'
OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';AUDIT=OUT/'audit';PAGES=OUT/'pages';VISUAL=OUT/'visuals';REVIEW=OUT/'review';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,AUDIT,PAGES,VISUAL,REVIEW,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)

## 2. Resolve completed Notebook 39 and upstream lineage

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def array_hash(a):
 a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def image_map(d):
 d=Path(d);return {p.stem:p for p in d.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if d.exists() else {}
def complete_runs(root,req):
 root=Path(root);return sorted([p for p in root.iterdir() if p.is_dir() and all((p/r).exists() for r in req)],key=lambda p:p.stat().st_mtime) if root.exists() else []
def write_json(p,x):Path(p).write_text(json.dumps(x,indent=2,sort_keys=True,default=str))
def loadz(p,keys):
 with np.load(p) as z:
  miss=[k for k in keys if k not in z.files]
  if miss:raise RuntimeError(f'{p}: missing {miss}')
  return {k:z[k].copy() for k in keys}

req39=['experiment_manifest.json','tables/semantic_structure_review.csv','tables/candidate_region_metrics.csv','audit/geometry_t80_replay.csv']
if SOURCE_NB39_RUN_OVERRIDE:NB39=Path(SOURCE_NB39_RUN_OVERRIDE)
else:
 hits=complete_runs(NB39_ROOT,req39)
 if not hits:raise RuntimeError(f'No complete Notebook 39 run below {NB39_ROOT}')
 NB39=hits[-1]
id39=json.loads((NB39/'experiment_manifest.json').read_text())
if id39.get('version') not in ['39-v8.2j','39-v8.2j-reconstruction-sensitivity']:raise RuntimeError(f'Unexpected Notebook 39 identity: {id39.get("version")}')
V82=Path(id39['source_v82']);V82C=Path(id39['source_v82c']);V75=Path(id39.get('v75',id39.get('v75_reference','')))
for p in [V82,V82C,V75]:
 if not p.exists():raise RuntimeError(f'Upstream path missing: {p}')
review=pd.read_csv(NB39/'tables'/'semantic_structure_review.csv').fillna('');regions=pd.read_csv(NB39/'tables'/'candidate_region_metrics.csv')
if review.region_id.duplicated().any() or regions.region_id.duplicated().any():raise RuntimeError('Duplicate Notebook 39 region IDs')
if review.structure_type.astype(str).str.strip().eq('').any():raise RuntimeError('Notebook 39 semantic review is incomplete')
reviewed=regions.merge(review[['region_id','structure_type','review_confidence','review_note']],on='region_id',validate='one_to_one')
# Only regions wholly within the primary .75-to-.50 interval.
reviewed=reviewed[(reviewed.higher_threshold<=HIGH_THRESHOLD)&(reviewed.lower_threshold>=LOW_THRESHOLD)].copy()
if reviewed.empty:raise RuntimeError('No reviewed Notebook 39 regions in the 0.75-to-0.50 interval')
SOURCE_FILES=[NB39/'experiment_manifest.json',NB39/'tables'/'semantic_structure_review.csv',NB39/'tables'/'candidate_region_metrics.csv',NB39/'audit'/'geometry_t80_replay.csv',V82/'experiment_identity.json',V82/'tables'/'all_eligible_oof_predictions.parquet',V82/'audit'/'input_and_label_audit.csv',V82C/'experiment_identity.json',V82C/'tables'/'all_eligible_label_sensitivity.parquet']
locks={str(p):sha256_file(p) for p in SOURCE_FILES};write_json(AUDIT/'source_file_hashes.json',locks)
print('Notebook 39:',NB39,'reviewed primary regions:',len(reviewed))

## 3. Semantic continuation-applicability estimand

In [ ]:
def applicability(st):
 if st in IMPLEMENTED_OLD:return 'IMPLEMENTED_OLD_STAFF_CONTINUATION'
 if st in PLAUSIBLE_UNIMPLEMENTED:return 'PLAUSIBLE_BUT_NOT_OLD_IMPLEMENTATION'
 if st in NOT_CONTINUATION:return 'NOT_A_CONTINUATION_TARGET'
 return 'NON_PRINT_MIXED_OR_INDETERMINATE'
reviewed['continuation_applicability']=reviewed.structure_type.map(applicability)
# selected_pixels from mutually exclusive adjacent probability bands supplies the pixel-weighted estimand.
app=(reviewed.groupby('continuation_applicability',as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'),source_groups=('source_group','nunique'),selected_pixels=('selected_pixels','sum')))
total=float(app.selected_pixels.sum());app['share_of_all_reviewed_pixels']=app.selected_pixels/total
semantic_den=float(app.loc[app.continuation_applicability!='NON_PRINT_MIXED_OR_INDETERMINATE','selected_pixels'].sum())
app['share_of_determinate_semantic_pixels']=np.where(app.continuation_applicability!='NON_PRINT_MIXED_OR_INDETERMINATE',app.selected_pixels/max(semantic_den,1),np.nan)
app.to_csv(TABLES/'continuation_applicability_summary.csv',index=False)
by_structure=(reviewed.groupby(['structure_type','continuation_applicability'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'),source_groups=('source_group','nunique'),selected_pixels=('selected_pixels','sum')))
by_structure['share_of_all_reviewed_pixels']=by_structure.selected_pixels/total;by_structure.to_csv(TABLES/'continuation_applicability_by_structure.csv',index=False)

# Page-cluster bootstrap of the broad plausible share, preserving page as the uncertainty unit.
page_class=reviewed.groupby(['page','continuation_applicability'],as_index=False).selected_pixels.sum();wide=page_class.pivot(index='page',columns='continuation_applicability',values='selected_pixels').fillna(0)
for c in ['IMPLEMENTED_OLD_STAFF_CONTINUATION','PLAUSIBLE_BUT_NOT_OLD_IMPLEMENTATION','NOT_A_CONTINUATION_TARGET','NON_PRINT_MIXED_OR_INDETERMINATE']:
 if c not in wide:wide[c]=0
category_cols=['IMPLEMENTED_OLD_STAFF_CONTINUATION','PLAUSIBLE_BUT_NOT_OLD_IMPLEMENTATION','NOT_A_CONTINUATION_TARGET','NON_PRINT_MIXED_OR_INDETERMINATE'];wide['total']=wide[category_cols].sum(axis=1);wide['plausible']=wide.IMPLEMENTED_OLD_STAFF_CONTINUATION+wide.PLAUSIBLE_BUT_NOT_OLD_IMPLEMENTATION
rng=np.random.default_rng(SEED);vals=[]
for _ in range(BOOTSTRAP_SAMPLES):
 q=wide.iloc[rng.integers(0,len(wide),len(wide))];vals.append(q.plausible.sum()/max(q.total.sum(),1))
point=float(wide.plausible.sum()/max(wide.total.sum(),1));ci=[float(np.percentile(vals,2.5)),float(np.percentile(vals,97.5))]
write_json(TABLES/'plausibly_continuable_pixel_share.json',{'numerator':'implemented old staff + plausible unimplemented stem/barline/beam/slur/tie/hairpin','denominator':'all reviewed selected pixels in 0.75-to-0.50 adjacent bands','point_estimate':point,'page_cluster_bootstrap_ci95':ci,'pages':len(wide),'bootstrap_samples':BOOTSTRAP_SAMPLES})
display(app);display(by_structure)

## 4. Resolve exact threshold selections and V7.5 reconstruction inputs

In [ ]:
pred=pd.read_parquet(V82/'tables'/'all_eligible_oof_predictions.parquet');pred['page']=pred.page.astype(str);pred['p_print']=pd.to_numeric(pred.p_print,errors='raise')
for c in ['y','x']:pred[c]=pd.to_numeric(pred[c],errors='raise').astype(int)
if pred.duplicated(['page','y','x']).any():raise RuntimeError('Duplicate prediction coordinates')
status=pd.read_csv(V75/'tables'/'cache_status.csv');status['page']=status.page.astype(str);CACHE={r.page:Path(r.cache) for _,r in status.iterrows()};CLEAN=image_map(ROOT/'source'/'clean');OBS=image_map(ROOT/'aligned');ia=pd.read_csv(V82/'audit'/'input_and_label_audit.csv');ia['page']=ia.page.astype(str);ia=ia.set_index('page')
pages=sorted(reviewed.page.astype(str).unique())

def select(g,shape,t):
 pm=np.zeros(shape,np.uint8);take=g.p_print.to_numpy()>=t;pm[g.y.to_numpy()[take],g.x.to_numpy()[take]]=1;n,lab,st,_=cv2.connectedComponentsWithStats(pm,8);out=np.zeros(shape,bool)
 for i in range(1,n):
  if st[i,cv2.CC_STAT_AREA]>=MIN_PRINT_RUN:out[lab==i]=1
 return out
runtime={};resolved=[]
for page in pages:
 if page not in CACHE or page not in CLEAN or page not in OBS or page not in ia.index:raise RuntimeError(f'{page}: missing upstream input')
 expected=[str(ia.loc[page,'clean_sha256']),str(ia.loc[page,'observed_sha256']),str(ia.loc[page,'cache_sha256'])];actual=[sha256_file(CLEAN[page]),sha256_file(OBS[page]),sha256_file(CACHE[page])]
 if expected!=actual:raise RuntimeError(f'{page}: upstream hash mismatch')
 clean=cv2.imread(str(CLEAN[page]),0);obs=cv2.imread(str(OBS[page]),0);a=loadz(CACHE[page],['mask','geometry_t80'])
 if not(clean.shape==obs.shape==a['mask'].shape):raise RuntimeError(f'{page}: shape mismatch')
 g=pred[pred.page==page];s75=select(g,obs.shape,HIGH_THRESHOLD);s50=select(g,obs.shape,LOW_THRESHOLD)
 if np.any(s75&~s50):raise RuntimeError(f'{page}: non-nested threshold selections')
 runtime[page]={'clean':clean,'obs':obs,'mask':a['mask']>0,'v78':a['geometry_t80'],'s75':s75,'s50':s50}
 resolved.append({'page':page,'clean_sha256':actual[0],'observed_sha256':actual[1],'cache_sha256':actual[2],'selected_075':int(s75.sum()),'selected_050':int(s50.sum()),'incremental_050_not_075':int((s50&~s75).sum())})
pd.DataFrame(resolved).to_csv(AUDIT/'resolved_inputs_and_threshold_counts.csv',index=False)
print('Resolved pages:',len(runtime))

## 5. Historical V5.3 reconstruction logic, preserved explicitly

In [ ]:
def immutable(original,candidate,mask):
 out=original.copy();out[mask>0]=candidate[mask>0]
 if not np.array_equal(out[mask==0],original[mask==0]):raise RuntimeError('Outside-mask drift')
 return out
def padded_components(mask,margin=CROP_MARGIN,min_area=8):
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);boxes=[];h,w=mask.shape
 for i in range(1,n):
  x,y,bw,bh,a=stats[i]
  if a>=min_area:boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
 changed=True
 while changed:
  changed=False;merged=[]
  while boxes:
   ax0,ay0,ax1,ay1=boxes.pop();rest=[]
   for bx0,by0,bx1,by1 in boxes:
    if not(ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
    else:rest.append((bx0,by0,bx1,by1))
   boxes=rest;merged.append((ax0,ay0,ax1,ay1))
  boxes=merged
 return boxes
def crop_inpaint(image,mask,method,margin=CROP_MARGIN):
 result=image.copy()
 for x0,y0,x1,y1 in padded_components(mask,margin):
  im=image[y0:y1,x0:x1];mk=np.where(mask[y0:y1,x0:x1]>0,255,0).astype(np.uint8)
  if method=='telea':cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA)
  elif method=='navier_stokes':cand=cv2.inpaint(im,mk,5,cv2.INPAINT_NS)
  elif method=='biharmonic':cand=np.clip(sk_inpaint.inpaint_biharmonic(im.astype(np.float32)/255,mk>0,channel_axis=None)*255,0,255).astype(np.uint8)
  else:raise ValueError(method)
  local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0];result[y0:y1,x0:x1]=local
 return immutable(image,result,mask)
def staff_evidence(observed,mask):
 visible=observed.copy();visible[mask>0]=255;ink=(visible<190).astype(np.uint8)*255;long=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((1,35),np.uint8));return cv2.dilate(long,np.ones((3,7),np.uint8))>0
def safe_staff_continuation(observed,mask,base,max_gap=MAX_STAFF_GAP,return_audit=False):
 out=base.copy();evidence=staff_evidence(observed,mask);h,w=observed.shape;rows=np.where(evidence.sum(axis=1)>=18)[0];audit=[]
 for y in rows:
  xs=np.where(mask[y]>0)[0]
  if not xs.size:continue
  for seg in np.split(xs,np.where(np.diff(xs)>1)[0]+1):
   if not len(seg) or len(seg)>max_gap:continue
   x0,x1=int(seg[0]),int(seg[-1]);r=16;left=evidence[max(0,y-1):min(h,y+2),max(0,x0-r):x0];right=evidence[max(0,y-1):min(h,y+2),x1+1:min(w,x1+r+1)]
   if left.sum()<2 or right.sum()<2:continue
   vals=[]
   for yy in range(max(0,y-1),min(h,y+2)):
    for xa,xb in [(max(0,x0-r),x0),(x1+1,min(w,x1+r+1))]:
     q=observed[yy,xa:xb];vals.extend(q[q<190].tolist())
   if not vals:continue
   value=min(int(np.percentile(vals,35)),140);band=mask[max(0,y-1):min(h,y+2),x0:x1+1]>0;patch=out[max(0,y-1):min(h,y+2),x0:x1+1];before=patch.copy();patch[band]=value;audit.append({'y':int(y),'x0':x0,'x1':x1,'gap_length':int(x1-x0+1),'written_pixels':int(np.sum((before!=patch)&band))})
 out=immutable(observed,out,mask)
 return (out,audit) if return_audit else out
def reconstruct(observed,mask):
 tele=crop_inpaint(observed,mask,'telea');ns=crop_inpaint(observed,mask,'navier_stokes');bi=crop_inpaint(observed,mask,'biharmonic');hc,hca=safe_staff_continuation(observed,mask,bi,return_audit=True);staff=staff_evidence(observed,mask);edge=cv2.dilate(cv2.Canny(observed,70,170),np.ones((3,3),np.uint8))>0;routed=bi.copy();route=(mask>0)&edge&~staff;routed[route]=ns[route];hr,hra=safe_staff_continuation(observed,mask,immutable(observed,routed,mask),return_audit=True)
 return {'telea':tele,'navier_stokes':ns,'biharmonic':bi,'hybrid_conservative':hc,'hybrid_routed':hr},{'hybrid_conservative':hca,'hybrid_routed':hra}
# Synthetic integrity test.
t=np.full((41,121),255,np.uint8);t[20,:]=60;m=np.zeros_like(t);m[18:23,50:71]=255;base=crop_inpaint(t,m,'telea');out,aud=safe_staff_continuation(t,m,base,return_audit=True);assert np.array_equal(out[m==0],t[m==0]);print('Historical-function self-test passed; continuation events:',len(aud))

## 6. Run V8.3 residual-mask benchmark at 0.75 and 0.50

In [ ]:
outputs={};page_rows=[];event_rows=[]
for page,z in runtime.items():
 folder=PAGES/page;folder.mkdir();cv2.imwrite(str(folder/'clean.png'),z['clean']);cv2.imwrite(str(folder/'observed.png'),z['obs']);cv2.imwrite(str(folder/'v7_8_geometry_t80.png'),z['v78'])
 outputs[page]={}
 for threshold,selected in [(HIGH_THRESHOLD,z['s75']),(LOW_THRESHOLD,z['s50'])]:
  residual=z['mask']&~selected;mask_u8=residual.astype(np.uint8)*255;methods,events=reconstruct(z['obs'],mask_u8)
  # Selected pixels are outside the residual mask and must remain exact observed bytes.
  for method,out in methods.items():
   if not np.array_equal(out[~z['mask']],z['obs'][~z['mask']]):raise RuntimeError(f'{page} {threshold} {method}: outside original mask drift')
   if not np.array_equal(out[selected],z['obs'][selected]):raise RuntimeError(f'{page} {threshold} {method}: selected bytes changed')
   cv2.imwrite(str(folder/f'p{int(threshold*100):02d}_{method}.png'),out)
   page_rows.append({'page':page,'threshold':threshold,'method':method,'selected_pixels':int(selected.sum()),'residual_pixels':int(residual.sum()),'outside_mask_drift':0,'selected_exact_observed':True,'output_sha256':array_hash(out)})
  for method,evs in events.items():
   for e in evs:event_rows.append({'page':page,'threshold':threshold,'method':method,**e})
  outputs[page][threshold]=methods
pd.DataFrame(page_rows).to_csv(TABLES/'v8_3_page_method_outputs.csv',index=False);pd.DataFrame(event_rows).to_csv(TABLES/'staff_continuation_events.csv',index=False)
print('Completed pages:',len(outputs),'events:',len(event_rows))

## 7. Score historical methods within reviewed Notebook 39 regions

In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):return cv2.Canny(im,70,170)>0
def f1(ref,pred,region,tol=1):
 ref=ref&region;pred=pred&region
 if not ref.any():return np.nan,np.nan,np.nan
 k=np.ones((3,3),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0;p=(pred&rd).sum()/max(pred.sum(),1);r=(ref&pd).sum()/ref.sum();return float(p),float(r),float(2*p*r/max(p+r,1e-12))
def metric(clean,out,region):
 diff=np.abs(clean.astype(np.float32)-out.astype(np.float32));mse=np.mean((clean[region].astype(float)-out[region].astype(float))**2);_,sm=ssim(clean,out,data_range=255,full=True);ip,ir,if1=f1(printed_ink(clean),printed_ink(out),region);_,_,ef1=f1(edges(clean),edges(out),region);sk=skeletonize(printed_ink(clean));near=cv2.dilate(printed_ink(out).astype(np.uint8),np.ones((3,3),np.uint8))>0
 return {'mae':float(diff[region].mean()),'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),'ssim':float(sm[region].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'skeleton_recall':float((sk&region&near).sum()/max((sk&region).sum(),1))}
score_rows=[]
for _,r in reviewed.iterrows():
 page=str(r.page);z=runtime[page];region=np.zeros(z['mask'].shape,bool);x0,y0,x1,y1=map(int,[r.x0,r.y0,r.x1,r.y1]);region[y0:y1,x0:x1]=True;region&=z['mask']
 for threshold in [HIGH_THRESHOLD,LOW_THRESHOLD]:
  for method,out in outputs[page][threshold].items():score_rows.append({'region_id':r.region_id,'page':page,'source_group':r.source_group,'structure_type':r.structure_type,'continuation_applicability':r.continuation_applicability,'threshold':threshold,'method':method,'selected_pixels':int(r.selected_pixels),**metric(z['clean'],out,region)})
scores=pd.DataFrame(score_rows);scores.to_csv(TABLES/'region_method_metrics.csv',index=False)
# Paired 0.50 minus 0.75 benefit: lower MAE is better; higher other metrics are better.
p=scores.pivot(index=['region_id','page','source_group','structure_type','continuation_applicability'],columns=['threshold','method'])
deltas=[]
for idx,row in p.iterrows():
 for method in METHODS:
  d={'region_id':idx[0],'page':idx[1],'source_group':idx[2],'structure_type':idx[3],'continuation_applicability':idx[4],'method':method}
  for k in ['mae','psnr','ssim','ink_recall','ink_f1','edge_f1','skeleton_recall']:
   a=row[(k,LOW_THRESHOLD,method)];b=row[(k,HIGH_THRESHOLD,method)];d[k+'_benefit_050_vs_075']=(b-a) if k=='mae' else (a-b)
  deltas.append(d)
delta=pd.DataFrame(deltas);delta.to_csv(TABLES/'paired_050_vs_075_method_deltas.csv',index=False)
summary=delta.groupby(['structure_type','continuation_applicability','method'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'),mae_benefit=('mae_benefit_050_vs_075','mean'),ink_recall_benefit=('ink_recall_benefit_050_vs_075','mean'),ink_f1_benefit=('ink_f1_benefit_050_vs_075','mean'),edge_f1_benefit=('edge_f1_benefit_050_vs_075','mean'),skeleton_recall_benefit=('skeleton_recall_benefit_050_vs_075','mean'))
summary.to_csv(TABLES/'v8_3_method_summary_by_structure.csv',index=False);display(summary)

## 8. Gap-span diagnostics and old-rule activation

In [ ]:
# Remaining horizontal and vertical run lengths after each threshold; semantic labels remain authoritative.
gap_rows=[]
for _,r in reviewed.iterrows():
 page=str(r.page);z=runtime[page];x0,y0,x1,y1=map(int,[r.x0,r.y0,r.x1,r.y1])
 for threshold,sel in [(HIGH_THRESHOLD,z['s75']),(LOW_THRESHOLD,z['s50'])]:
  residual=z['mask']&~sel;local=residual[y0:y1,x0:x1];hr=[];vr=[]
  for row in local:
   xs=np.where(row)[0]
   hr.extend([len(q) for q in np.split(xs,np.where(np.diff(xs)>1)[0]+1) if len(q)]) if len(xs) else None
  for col in local.T:
   ys=np.where(col)[0]
   vr.extend([len(q) for q in np.split(ys,np.where(np.diff(ys)>1)[0]+1) if len(q)]) if len(ys) else None
  gap_rows.append({'region_id':r.region_id,'page':page,'structure_type':r.structure_type,'continuation_applicability':r.continuation_applicability,'threshold':threshold,'residual_pixels':int(local.sum()),'max_horizontal_gap':max(hr) if hr else 0,'median_horizontal_gap':float(np.median(hr)) if hr else 0,'max_vertical_gap':max(vr) if vr else 0,'median_vertical_gap':float(np.median(vr)) if vr else 0})
gaps=pd.DataFrame(gap_rows);gaps.to_csv(TABLES/'residual_gap_spans_by_region.csv',index=False)
wide_g=gaps.pivot(index=['region_id','page','structure_type','continuation_applicability'],columns='threshold',values=['residual_pixels','max_horizontal_gap','max_vertical_gap']).reset_index();wide_g.to_csv(TABLES/'residual_gap_span_comparison.csv',index=False)
print(gaps.groupby(['structure_type','threshold'])[['residual_pixels','max_horizontal_gap','max_vertical_gap']].mean())

## 9. Visual summaries and targeted comparison sheets

In [ ]:
q=app.sort_values('share_of_all_reviewed_pixels');plt.figure(figsize=(10,5));plt.barh(q.continuation_applicability,q.share_of_all_reviewed_pixels);plt.xlabel('Share of reviewed 0.50-but-not-0.75 pixels');plt.tight_layout();plt.savefig(VISUAL/'continuation_applicability_share.png',dpi=180);plt.close()
# Method benefit chart.
chart=summary.groupby('method',as_index=False).mae_benefit.mean().sort_values('mae_benefit');plt.figure(figsize=(9,5));plt.barh(chart.method,chart.mae_benefit);plt.axvline(0,color='black');plt.xlabel('Mean local MAE benefit: 0.50 over 0.75');plt.tight_layout();plt.savefig(VISUAL/'method_mae_benefit_050_vs_075.png',dpi=180);plt.close()
# Curated sheets for largest absolute effects.
chosen=delta.assign(abs_gain=delta.mae_benefit_050_vs_075.abs()).sort_values('abs_gain',ascending=False).drop_duplicates('region_id').head(80)
for _,r in chosen.iterrows():
 rr=reviewed.set_index('region_id').loc[r.region_id];page=str(r.page);z=runtime[page];x0,y0,x1,y1=map(int,[rr.x0,rr.y0,rr.x1,rr.y1]);pad=12;x0=max(0,x0-pad);y0=max(0,y0-pad);x1=min(z['obs'].shape[1],x1+pad);y1=min(z['obs'].shape[0],y1+pad)
 ims=[z['clean'][y0:y1,x0:x1],z['obs'][y0:y1,x0:x1],outputs[page][HIGH_THRESHOLD]['hybrid_routed'][y0:y1,x0:x1],outputs[page][LOW_THRESHOLD]['hybrid_routed'][y0:y1,x0:x1]];names=['clean','observed','0.75 hybrid','0.50 hybrid'];fig,ax=plt.subplots(1,4,figsize=(16,4))
 for a,im,n in zip(ax,ims,names):a.imshow(im,cmap='gray',vmin=0,vmax=255);a.set_title(n);a.axis('off')
 fig.suptitle(f'{r.region_id} | {r.structure_type}');plt.tight_layout();plt.savefig(REVIEW/f'{r.region_id}.png',dpi=150);plt.close(fig)
print('Visuals:',VISUAL,'Review:',REVIEW)

## 10. Manifest, interpretation and final upload package

In [ ]:
current={str(p):sha256_file(p) for p in SOURCE_FILES}
if current!=json.loads((AUDIT/'source_file_hashes.json').read_text()):raise RuntimeError('Source artefact changed during V8.3 run')
implemented=float(app.loc[app.continuation_applicability=='IMPLEMENTED_OLD_STAFF_CONTINUATION','share_of_all_reviewed_pixels'].sum());plausible=float(app.loc[app.continuation_applicability.isin(['IMPLEMENTED_OLD_STAFF_CONTINUATION','PLAUSIBLE_BUT_NOT_OLD_IMPLEMENTATION']),'share_of_all_reviewed_pixels'].sum())
lines=['NOTEBOOK 40 — V8.3 RESIDUAL-GAP CONTINUATION FEASIBILITY','',f'Notebook 39 source: {NB39}',f'Reviewed primary regions: {len(reviewed)}',f'Reviewed selected pixels: {int(total)}',f'Historical implementation coverage (staff only): {implemented:.6f}',f'Broad semantically plausible continuation share: {plausible:.6f}',f'Page-clustered 95% bootstrap interval for broad share: [{ci[0]:.6f}, {ci[1]:.6f}]','','INTERPRETATION','Historical code directly tests horizontal staff continuation only.','Stem/barline, beam, slur/tie and hairpin shares are opportunity estimates, not implemented completion claims.','Method outputs compare 0.75 and 0.50 residual masks against clean truth after reconstruction.','No result changes the registered threshold or authorises production use.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines));print(chr(10).join(lines))
manifest={'version':'8.3-development-residual-gap-continuation','notebook':40,'created_utc':datetime.now(timezone.utc).isoformat(),'development_only':True,'independent_data_used':False,'production_authority':False,'source_notebook39':str(NB39),'source_v82':str(V82),'source_v82c':str(V82C),'source_v75':str(V75),'thresholds':[HIGH_THRESHOLD,LOW_THRESHOLD],'methods':METHODS,'historical_staff_constants':{'horizontal_open_width':35,'evidence_dilation':[3,7],'max_gap':MAX_STAFF_GAP,'flank_radius':16,'dark_percentile':35,'value_cap':140},'semantic_categories':{'implemented_old':sorted(IMPLEMENTED_OLD),'plausible_unimplemented':sorted(PLAUSIBLE_UNIMPLEMENTED),'not_continuation':sorted(NOT_CONTINUATION),'non_target':sorted(NON_TARGET)},'source_hashes':current,'software':{'python':sys.version,'platform':platform.platform(),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__}}
write_json(OUT/'experiment_manifest.json',manifest)
arts=[OUT/'executive_summary.txt',OUT/'experiment_manifest.json',TABLES/'continuation_applicability_summary.csv',TABLES/'continuation_applicability_by_structure.csv',TABLES/'plausibly_continuable_pixel_share.json',TABLES/'v8_3_page_method_outputs.csv',TABLES/'staff_continuation_events.csv',TABLES/'region_method_metrics.csv',TABLES/'paired_050_vs_075_method_deltas.csv',TABLES/'v8_3_method_summary_by_structure.csv',TABLES/'residual_gap_spans_by_region.csv',TABLES/'residual_gap_span_comparison.csv',AUDIT/'source_file_hashes.json',AUDIT/'resolved_inputs_and_threshold_counts.csv',VISUAL/'continuation_applicability_share.png',VISUAL/'method_mae_benefit_050_vs_075.png']
miss=[str(p) for p in arts if not p.exists()]
if miss:raise RuntimeError(f'Missing upload artefacts: {miss}')
for p in arts:
 d=UPLOAD/p.relative_to(OUT);d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,d)
if REVIEW.exists():shutil.copytree(REVIEW,UPLOAD/'review',dirs_exist_ok=True)
(UPLOAD/'README.txt').write_text('Notebook 40 / V8.3 development-only residual-gap continuation audit. Old code implements staff continuation only; other directional structures are feasibility categories, not implemented claims.'+chr(10))
inv=[]
for p in sorted(UPLOAD.rglob('*')):
 if p.is_file():inv.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in inv));bundle=OUT/'notebook40_v8_3_residual_gap_continuation_upload_bundle.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file():z.write(p,p.relative_to(UPLOAD))
print('Bundle:',bundle)